In [1]:
# Add at the top, before imports
import os
import shutil
cache_dir = "/root/.cache/huggingface"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("✅ Cleared HuggingFace cache")

In [2]:
# Cell 1 – RUN THIS AND RESTART KERNEL AFTER
import shutil, os, torch
from pathlib import Path

# 1. Clear Unsloth compiled cache (fixes the 'int.mean' bug 95% of the time)
for cache_path in ["/root/.cache/huggingface", "/kaggle/working/unsloth_compiled_cache"]:
    if os.path.exists(cache_path):
        shutil.rmtree(cache_path)
        print(f"Deleted {cache_path}")

# 2. Clear PyTorch + CUDA cache completely
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.ipc_collect()

# 3. Clear Kaggle temp files that sometimes break Unsloth
!rm -rf /tmp/torch_extensions /tmp/torchinductor_* 2>/dev/null || true

print("All caches cleared! Now click Runtime → Restart Session (or the restart button) and run the next cells fresh.")

All caches cleared! Now click Runtime → Restart Session (or the restart button) and run the next cells fresh.


In [3]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-xek2oq_r/unsloth_1ca4ffe3881b41609234cd26390f50f4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-xek2oq_r/unsloth_1ca4ffe3881b41609234cd26390f50f4
  Resolved https://github.com/unslothai/unsloth.git to commit d4a311d8e71692961e5da1d26d98197fde94f41a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.4/284.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.1 MB/s eta 0:00

In [4]:
import unsloth
import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-11-26 14:05:12.806482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764165912.977572      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764165913.022406      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[xformers|WARNING]WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.9 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.9 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
# ============================================
# STEP 1: Load Model
# ============================================
max_seq_length = 1024  # ✅ This is YOUR training context length, NOT pretraining!
                       # Qwen 2.5 Coder was pretrained with 32K context
                       # But we train at 1024 for speed and memory efficiency

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-bnb-4bit",  # ✅ CHANGED from Llama 8B
    max_seq_length=max_seq_length,  # 1024 - YOUR practical training limit
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # ✅ LoRA rank (keep as is)
    lora_alpha=16,           # ✅ LoRA alpha (keep as is)
    lora_dropout=0,          # ✅ No dropout (keep as is)
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],  # ✅ Keep same
    use_rslora=True,         # ✅ Rank-stabilized LoRA (keep as is)
    use_gradient_checkpointing="unsloth"  # ✅ Memory optimization (keep as is)
)

==((====))==  Unsloth 2025.11.4: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 6.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
# ============================================
# STEP 2: Setup Tokenizer with Chat Template
# ============================================
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # ✅ CHANGED: Qwen uses ChatML format, not Llama-3
)

# actually llama3  ko tokenizer : eostoken=padtoken by default 
# here no its not same by default in qwen
# have a loook at this : solution https://github.com/unslothai/unsloth/issues/416
# Ensure padding token is set
# tokenizer.pad_token = tokenizer.eos_token

Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


In [7]:
def format_codealpaca(examples):
    conversations = []
    for instruction, input_text, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):
        if input_text and input_text.strip():
            user_message = f"{instruction}\n\nInput:\n{input_text}"
        else:
            user_message = instruction
        conversation = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": output}
        ]
        conversations.append(conversation)
    
    # Apply chat template and REMOVE trailing whitespace
    texts = [
        tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False).rstrip()
        for conv in conversations
    ]
   
    return {"text": texts}

In [8]:
dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
dataset = dataset.train_test_split(train_size=7000, test_size=1500, seed=42)
train_dataset = dataset["train"]  # 7000 examples
test_dataset = dataset["test"]    # 1500 examples
train_dataset = train_dataset.map(format_codealpaca, batched=True, remove_columns=train_dataset.column_names)
test_dataset = test_dataset.map(format_codealpaca, batched=True, remove_columns=test_dataset.column_names)

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [9]:
# Check what's inside a checkpoint folder
checkpoint_path = "/kaggle/input/codingchatbot/checkpoints/checkpoint-438"
import os
if os.path.exists(checkpoint_path):
    print("✅ Checkpoint exists!")
    print("\nFiles in checkpoint:")
    for file in os.listdir(checkpoint_path):
        print(f"  - {file}")
else:
    print("❌ Checkpoint not found!")

✅ Checkpoint exists!

Files in checkpoint:
  - adapter_model.safetensors
  - merges.txt
  - trainer_state.json
  - training_args.bin
  - adapter_config.json
  - README.md
  - tokenizer.json
  - vocab.json
  - tokenizer_config.json
  - scaler.pt
  - chat_template.jinja
  - scheduler.pt
  - special_tokens_map.json
  - optimizer.pt
  - rng_state.pth
  - added_tokens.json


In [10]:
# ============================================
# STEP 5: Training Configuration
# ============================================
import os
from transformers import TrainingArguments
from trl import SFTTrainer

# Kaggle output directory
output_dir = "/kaggle/working/checkpoints2"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.makedirs(output_dir, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  # ✅ CHANGED: Add eval dataset (you have it!)
    dataset_text_field="text",
    max_seq_length=1024,
    dataset_num_proc=2,
    packing=False,  # Keep off for stability
    args=TrainingArguments(
        output_dir=output_dir,
        
        # ✅ CHANGED: Increase batch size (1.5B is smaller!)
        per_device_train_batch_size=2,        # ← 2→4 (you have more memory now!)
        gradient_accumulation_steps=8,        # ← 8→4 (4×4 = effective batch 16)
        
        # ✅ CHANGED: Slightly lower learning rate for smaller model
        learning_rate=3e-4,                   # ← 2e-4 → 3e-4 (smaller models train faster)
        
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        num_train_epochs=1,
        
        fp16=True,
        bf16=False,
        
        logging_steps=10,
        logging_first_step=True,
        
        # ✅ CHANGED: Enable evaluation
        eval_strategy="steps",                # ← Track eval loss
        eval_steps=120,                       # ← Evaluate every 120 steps
        
        save_strategy="steps",
        save_steps=120,
        save_total_limit=2,
        
        # ✅ CHANGED: Enable best model selection
        load_best_model_at_end=True,         # ← Save best checkpoint
        metric_for_best_model="eval_loss",   # ← Use eval loss as metric
        
        warmup_ratio=0.03,
        weight_decay=0.01,
        dataloader_num_workers=2,
        
        torch_compile=False,
        gradient_checkpointing=True,
        
        report_to="none",
    ),
    dataset_kwargs={
        "append_concat_token": False,
        "add_special_tokens": False,
    },
)

# ============================================
# STEP 6: Train and Save
# ============================================
trainer.train(resume_from_checkpoint=checkpoint_path)

# Save final model
final_model_dir = "/kaggle/working/final_model2"
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ Training complete! Model saved to:", final_model_dir)

Map (num_proc=2):   0%|          | 0/7000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 1 | Total steps = 438
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Could not locate the best model at /kaggle/working/checkpoints/checkpoint-360/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


Step,Training Loss,Validation Loss


✅ Training complete! Model saved to: /kaggle/working/final_model2
